# 04 - Experience C : adaptation du detecteur (H3)

Hypothese H3 : adapter le detecteur restaure une part de la performance perdue a cause du decalage de
domaine. On compare quatre strategies sur une meme partition de test, pour que la comparaison soit juste :

1. detecteur naturel seul, applique aux sources generees, qui sert de rappel du probleme ;
2. reentrainement mixte, un seul detecteur entraine sur naturel et genere melanges ;
3. detection consciente de la source, on identifie d'abord la source puis on applique le detecteur correspondant ;
4. detecteur apparie, entraine et teste sur la meme source, qui donne la borne haute atteignable.

Un dernier volet teste la generalisation a un generateur non vu a l'entrainement. Les caracteristiques
SRM sont deja extraites et deposees sur Drive. Figures nommees expC_srm_<type>_<algo>_p<charge>.png.

## Configuration

In [ ]:
# ================= CONFIGURATION =================
FEATURE   = 'srm'
ALGOS     = ['lsb']            # la solution vise le decalage de domaine, lisible surtout sur LSB
PAYLOAD   = 0.4
NATUREL   = 'natural'
GENERES   = ['sd', 'sdxl', 'adm']
N_PCA     = 300               # meme reduction que l'experience A
TEST_SIZE = 0.3               # part reservee au test, fixee pour toutes les strategies
SEED      = 42
# ================================================
SOURCES = [NATUREL] + GENERES
pp = str(PAYLOAD).replace('.', '')
print('Experience C sur', FEATURE, '| algos', ALGOS, '| charge', PAYLOAD)

In [ ]:
import os
import numpy as np
np.random.seed(SEED)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/memoire_data'
except Exception:
    DATA_DIR = os.path.abspath('./memoire_data')
FEAT_DIR = f'{DATA_DIR}/features_{FEATURE}'
FIG = f'{DATA_DIR}/figures'
os.makedirs(FIG, exist_ok=True)
print('Caracteristiques :', FEAT_DIR, '| Figures :', FIG)

In [ ]:
!pip install -q scikit-learn matplotlib
print('Installation terminee.')

## 1. Partition commune et detecteur

Pour chaque source on separe une fois pour toutes un jeu d'entrainement et un jeu de test, avec la meme
graine. Toutes les strategies s'entrainent sur les parties train et sont evaluees sur les memes parties
test, ce qui rend les AUC directement comparables. Le detecteur est celui de l'experience A : mise a
l'echelle, reduction PCA, puis regression logistique.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

def charger(source, setname):
    return np.load(f'{FEAT_DIR}/{source}__{setname}.npy')

def detecteur():
    return make_pipeline(StandardScaler(),
                         PCA(n_components=N_PCA, random_state=SEED),
                         LogisticRegression(max_iter=5000))

def partition(source, algo):
    # jeu cover contre stego d'une source, puis separation train et test fixe
    Xc = charger(source, 'cover')
    Xs = charger(source, f'{algo}_p{PAYLOAD}')
    n = min(len(Xc), len(Xs)); Xc, Xs = Xc[:n], Xs[:n]
    X = np.vstack([Xc, Xs]); y = np.concatenate([np.zeros(n), np.ones(n)])
    return train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)

print('Detecteur SRM pret, PCA a', N_PCA, 'composantes.')

## 2. Les quatre strategies

On calcule, pour chaque algorithme et chaque source generee, l'AUC de test sous les quatre strategies. La
detection consciente de la source repose sur un classifieur de source, entraine a reconnaitre l'origine
d'une image ; vu l'ampleur du decalage de domaine, il l'identifie tres bien, et l'on route alors chaque
image vers le detecteur apparie de sa source predite.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

resultats = {}
for algo in ALGOS:
    # partitions fixes par source
    part = {s: partition(s, algo) for s in SOURCES}
    Xtr = {s: part[s][0] for s in SOURCES}; Xte = {s: part[s][1] for s in SOURCES}
    ytr = {s: part[s][2] for s in SOURCES}; yte = {s: part[s][3] for s in SOURCES}

    # strategie 1 : detecteur naturel seul
    det_nat = detecteur().fit(Xtr[NATUREL], ytr[NATUREL])
    auc_nat = {g: roc_auc_score(yte[g], det_nat.predict_proba(Xte[g])[:, 1]) for g in GENERES}

    # strategie 2 : reentrainement mixte sur toutes les sources melangees
    Xmix = np.vstack([Xtr[s] for s in SOURCES]); ymix = np.concatenate([ytr[s] for s in SOURCES])
    det_mix = detecteur().fit(Xmix, ymix)
    auc_mix = {g: roc_auc_score(yte[g], det_mix.predict_proba(Xte[g])[:, 1]) for g in GENERES}

    # strategie 4 : detecteur apparie, borne haute
    det_app = {s: detecteur().fit(Xtr[s], ytr[s]) for s in SOURCES}
    auc_app = {g: roc_auc_score(yte[g], det_app[g].predict_proba(Xte[g])[:, 1]) for g in GENERES}

    # strategie 3 : detection consciente de la source
    # classifieur de source entraine sur les images d'entrainement, toutes classes confondues
    Xsrc = np.vstack([Xtr[s] for s in SOURCES])
    ysrc = np.concatenate([[i] * len(Xtr[s]) for i, s in enumerate(SOURCES)])
    clf_src = make_pipeline(StandardScaler(), PCA(n_components=N_PCA, random_state=SEED),
                            LogisticRegression(max_iter=5000)).fit(Xsrc, ysrc)
    auc_aware = {}
    acc_src = {}
    for g in GENERES:
        pred = clf_src.predict(Xte[g])
        acc_src[g] = float(np.mean(pred == SOURCES.index(g)))
        # routage par lots : on regroupe les images selon la source predite, un score par groupe
        scores = np.empty(len(Xte[g]))
        for si, s in enumerate(SOURCES):
            masque = pred == si
            if masque.any():
                scores[masque] = det_app[s].predict_proba(Xte[g][masque])[:, 1]
        auc_aware[g] = roc_auc_score(yte[g], scores)

    resultats[algo] = dict(nat=auc_nat, mix=auc_mix, aware=auc_aware, app=auc_app, acc_src=acc_src)

    print(f'\n=== {algo} {PAYLOAD} bpp ===')
    print(f"{'source':6s} {'nat seul':>9s} {'mixte':>7s} {'conscient':>10s} {'apparie':>8s}  id source")
    for g in GENERES:
        print(f'{g:6s} {auc_nat[g]:9.3f} {auc_mix[g]:7.3f} {auc_aware[g]:10.3f} {auc_app[g]:8.3f}   {acc_src[g]:.3f}')

## 3. Figure : les strategies par source

Pour chaque algorithme, un graphe compare les quatre strategies sur les sources generees. La barre du
detecteur naturel seul rappelle le probleme, les barres du reentrainement mixte et de la detection
consciente montrent la part de performance restauree, et le detecteur apparie marque la borne haute.

In [ ]:
import matplotlib.pyplot as plt

for algo in ALGOS:
    R = resultats[algo]
    strategies = [('naturel seul', 'nat', '#e76f51'),
                  ('mixte', 'mix', '#f4a261'),
                  ('conscient de la source', 'aware', '#2a9d8f'),
                  ('apparie', 'app', '#264653')]
    x = np.arange(len(GENERES)); largeur = 0.2
    fig, ax = plt.subplots(figsize=(9, 5))
    for k, (nom, cle, coul) in enumerate(strategies):
        vals = [R[cle][g] for g in GENERES]
        ax.bar(x + (k - 1.5) * largeur, vals, largeur, label=nom, color=coul)
    ax.axhline(0.5, ls='--', color='gray')
    ax.set_xticks(x); ax.set_xticklabels(GENERES); ax.set_ylim(0.4, 1.0)
    ax.set_ylabel('AUC de test'); ax.legend()
    ax.set_title(f'Adaptation du detecteur, {algo} {PAYLOAD} bpp, SRM')
    nom = f'{FIG}/expC_srm_strategies_{algo}_p{pp}.png'
    plt.tight_layout(); plt.savefig(nom, dpi=150); plt.show()
    print('figure :', os.path.basename(nom))

## 4. Generalisation a un generateur non vu

Le reentrainement mixte suppose que l'on dispose deja d'exemples de la source visee. On teste donc un cas
plus exigeant : entrainer sur le naturel et deux generateurs, puis evaluer sur le troisieme, jamais vu.
On compare au detecteur naturel seul sur cette meme source. Si l'AUC remonte, c'est que voir du contenu
genere, meme d'autres modeles, aide a affronter un generateur inconnu.

In [ ]:
loso = {}
for algo in ALGOS:
    part = {s: partition(s, algo) for s in SOURCES}
    Xtr = {s: part[s][0] for s in SOURCES}; Xte = {s: part[s][1] for s in SOURCES}
    ytr = {s: part[s][2] for s in SOURCES}; yte = {s: part[s][3] for s in SOURCES}
    det_nat = detecteur().fit(Xtr[NATUREL], ytr[NATUREL])
    ligne = {}
    for held in GENERES:
        vus = [NATUREL] + [g for g in GENERES if g != held]
        Xt = np.vstack([Xtr[s] for s in vus]); yt = np.concatenate([ytr[s] for s in vus])
        det = detecteur().fit(Xt, yt)
        auc_vu = roc_auc_score(yte[held], det.predict_proba(Xte[held])[:, 1])
        auc_seul = roc_auc_score(yte[held], det_nat.predict_proba(Xte[held])[:, 1])
        ligne[held] = (auc_seul, auc_vu)
    loso[algo] = ligne
    print(f'\n=== {algo} {PAYLOAD} bpp, generateur non vu ===')
    for held in GENERES:
        a, b = ligne[held]
        print(f'  {held:5s} naturel seul {a:.3f}  ->  mixte sans {held} {b:.3f}')

In [ ]:
for algo in ALGOS:
    ligne = loso[algo]
    x = np.arange(len(GENERES)); largeur = 0.35
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - largeur/2, [ligne[g][0] for g in GENERES], largeur, label='naturel seul', color='#e76f51')
    ax.bar(x + largeur/2, [ligne[g][1] for g in GENERES], largeur, label='mixte, source non vue', color='#2a9d8f')
    ax.axhline(0.5, ls='--', color='gray')
    ax.set_xticks(x); ax.set_xticklabels(GENERES); ax.set_ylim(0.4, 1.0)
    ax.set_ylabel('AUC sur la source non vue'); ax.legend()
    ax.set_title(f'Generalisation a un generateur non vu, {algo} {PAYLOAD} bpp, SRM')
    nom = f'{FIG}/expC_srm_loso_{algo}_p{pp}.png'
    plt.tight_layout(); plt.savefig(nom, dpi=150); plt.show()
    print('figure :', os.path.basename(nom))

## Suite

On copie les figures dans le dossier results du depot et on consigne les chiffres dans
results/experience_C.md, avec l'interpretation de chaque figure dans results/figures_interpretations.md.
Lecture attendue : la solution restaure ce que le decalage de domaine avait fait perdre, sans recuperer
ce que l'interference masque ni ce qui manque faute de donnees pour l'adaptatif.